## Task 1: Multi-Object Based Synopsis
- background modeling
- object detection
- euclidean distance tracker
- 

In [ ]:
# task 1: multi-object base synopsis

# imports:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
#background modeling:

frame_count = 0

# function for extracting frames from the videos and returns the grey frames in an array
def extract_frames(video_path):
  '''takes the filepath of the video and extracts its frames. Returns an array of grey frames '''
  cap = cv2.VideoCapture(video_path)

  # Check if video opened successfully
  if not cap.isOpened():
    print("Error opening video file")

  # get video details for output
  frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  fps = cap.get(cv2.CAP_PROP_FPS)
  frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
  print(f"frame size: {frame_width}x{frame_height}")
  print(f"fps:{fps}")
  print(f"frame count:{frame_count}")

  count = 0
  frames = [] # initialize array
  # open the video and loop thru the frames to extract them
  print("looping thru frames to extract:")
  while(cap.isOpened()):
    # print("inside loop")
    ret, frame = cap.read() # read the frame
    if ret == True:
      print(f"extracting frame number {count}")
      frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) # make sure only one grey channel instead of rgb
      frames.append(frame) # save the frame into list
      count += 1
    else:
          print("End of video or error occurred.")
          break
  cap.release()
  return frames

# function to get BG0 given the frames
def get_BG0(frames, step=5):
  ''' median of every step-th frame, so moving objects disappear '''
  sample = np.stack(frames[::step], axis=0)
  return np.median(sample, axis=0).astype(np.uint8)


# FGit = It ? |BGit+1 - It| > T : 0 OR FGit = It ? |BGit - It| > T : 0
# function to calculate FG from curr frame and next BG OR curr BG
def foreground(It, BGt_1):
  '''takes the curr frame and the curr background model. Returns the foreground (FG) and the mask(Msk)'''
  rows, cols = It.shape
  FG = np.zeros((rows,cols), dtype = np.uint8) # create empty foreground
  Msk = np.zeros((rows,cols), dtype = np.uint8) # create empty Mask
  thresh = 30 # edited
  # for i in range(rows):
  #   for j in range(cols):
  #     diff = BGt_1[i,j].astype(np.int16) - It[i,j].astype(np.int16)
  #     diff = np.abs(diff) # get abs diff betw next background and curr frame
  #     if diff > thresh: # check diff with threshold
  #       FG[i,j] = It[i,j]
  #       Msk[i,j] = 255
  #     else:
  #       FG[i,j] = 0
  #       Msk[i,j] = 0
  diff = np.abs(BGt_1.astype(np.int16)-It.astype(np.int16))
  FG = np.where((diff > thresh), It, 0).astype(np.uint8)
  Msk = np.where((diff > thresh), 255, 0).astype(np.uint8)
  return FG, Msk

# function to update background model -  need to use Msk instead of beta
def background_beta(It, BGt, beta):
  '''takes the curr frame, the curr background model, and beta. Calculates next background model '''
  rows, cols = It.shape
  BGt_1 = np.zeros((rows,cols), dtype = np.uint8) # create empty next BG model
  # for i in range(rows):
  #   for j in range(cols):
  #     w_sum = (1-beta) * BGt[i,j].astype(np.int16) + beta * It[i,j].astype(np.int16) # calculate weighted sum using beta
  #     BGt_1[i,j] = round(w_sum) # convert to int using roundign and put into next BG model
  w_sum = (1- beta) * BGt.astype(np.float32) + beta * It.astype(np.float32)
  BGt_1 = np.round(w_sum).astype(np.uint8)
  return BGt_1

# alternative, uses the background model creator that doesn't use masks
# function to create the foreground images, the corresponding mask images, and updates the background model over all frames
def moving_average_BG_alternate(frames, BGt, beta, path):
  ''' takes the frames, beta, and BG0 (BGt) to get the foreground, mask, and updated background models for each frame'''

  c = len(frames)
  for i in range(1,c): # loop over frames starting from second frame
    frame = frames[i] # get the curr frame
    # BGt_1 = background(frame, BGt, beta) # next background model
    # CHECK THIS!!!!!
    # FG, Msk = foreground(frame, BGt_1) # foreground using next background
    FG, Msk = foreground(frame, BGt) # foreground using curr background
    BGt_1 = background_beta(frame, BGt, beta) # next background model


    # write as images
    cv2.imwrite(f"{path}/Background/background_alternate_{i:04}.png",BGt_1)
    cv2.imwrite(f"{path}/Foreground/foreground_alternate_{i:04}.png",FG)
    cv2.imwrite(f"{path}/Mask/mask_alternate_{i:04}.png",Msk)
    BGt = BGt_1.copy() # set curr BG model for next iteration as the next background model from this iteration

def writeVideo(output_path, frames_path, fps=30):
    ''' used to create output videos from the frames'''  
    # create codec for writing the video
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    # create output for writing the grey video
    cap = cv2.VideoCapture(frames_path) # sequence of images

    if not cap.isOpened():
        print(f"Error: Could not open image sequence at {frames_path}")
        return

    # 2. Get dimensions automatically from the first image frame
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"frame size: {frame_width}x{frame_height}")
    print(f"fps:{fps}")
    print(f"frame count:{frame_count}")

    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height), isColor=True)


    count = 0
    # loop through the image sequence and write into the out vid
    print("looping through image sequence to write vid")
    while(cap.isOpened()):
        # print("inside writing video loop")
        ret, frame = cap.read()
        if ret == True:
            out.write(frame)
            print(f"writing frame number {count}")
            count += 1
        else:
                print("End of video or error occurred.")
                break

    cap.release()
    out.release()

beta = 0.02 # edited

# test on video
frames1 = extract_frames('tabletennis_short.mp4')

BG0_1 = get_BG0(frames1) # prev - BGt
cv2.imwrite(f"Results/Background/background_alternate_0000.png",BG0_1)

# using only beta for background model
moving_average_BG_alternate(frames1, BG0_1, beta, 'Results' )
writeVideo('tabletennis_foregrounds_alternate.mp4', 'Results/Foreground/foreground_alternate_%04d.png')
writeVideo('tabletennis_backgrounds_alternate.mp4', 'Results/Background/background_alternate_%04d.png')
writeVideo('tabletennis_masks_alternate.mp4', 'Results/Mask/mask_alternate_%04d.png')
print("saved videos")



# object detection: background subtraction + morphology + connected components

def clean_mask(Msk, open_k=3, close_k=21, dilate_iter=3):
    '''takes the raw mask from foreground() and cleans it with morphology'''
    Msk = Msk.astype(np.uint8)
    k_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_k, open_k))
    k_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k))
    Msk = cv2.morphologyEx(Msk, cv2.MORPH_OPEN, k_small)    # remove specks
    Msk = cv2.morphologyEx(Msk, cv2.MORPH_CLOSE, k_big)     # fill holes
    Msk = cv2.dilate(Msk, k_small, iterations=dilate_iter)  # merge fragments
    return Msk

def merge_boxes(dets, gap=25):
    '''merges boxes that overlap or are within `gap` pixels of each other'''
    boxes = [[x, y, x+w, y+h] for (x, y, w, h, cx, cy) in dets]
    merged = True
    while merged:
        merged = False
        out = []
        while boxes:
            a = boxes.pop()
            i = 0
            while i < len(boxes):
                b = boxes[i]
                if (a[0] - gap < b[2] and b[0] - gap < a[2] and
                    a[1] - gap < b[3] and b[1] - gap < a[3]):
                    a = [min(a[0], b[0]), min(a[1], b[1]),
                         max(a[2], b[2]), max(a[3], b[3])]
                    boxes.pop(i)
                    merged = True
                else:
                    i += 1
            out.append(a)
        boxes = out
    return [(x1, y1, x2-x1, y2-y1, (x1+x2)//2, (y1+y2)//2) for x1, y1, x2, y2 in boxes]

def detect_objects(frame, BG, min_area=1500):
    '''takes a grey frame and the background model.
    Returns a list of (x, y, w, h, cx, cy) per blob, and the cleaned mask'''
    _, Msk = foreground(frame, BG)
    Msk = clean_mask(Msk)
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(Msk, connectivity=8)
    detections = []
    for i in range(1, n):  # label 0 is the background
        x, y, w, h, area = stats[i]
        if area >= min_area:
            cx, cy = centroids[i]
            detections.append((int(x), int(y), int(w), int(h), int(cx), int(cy)))
    return merge_boxes(detections), Msk   # <- changed: merge before returning


# euclidean distance tracker and synopsis generation


# Create the Synopsis folder (exist_ok=True means it won't error if it's already there)
os.makedirs('Results/Synopsis', exist_ok=True)

class Track:
    """One tracked object: id, last centre, last frame seen, trajectory, and the crop."""
    _next_id = 1
    def __init__(self, cx, cy, frame_idx, crop=None):
        self.id = Track._next_id
        Track._next_id += 1
        self.cx, self.cy = float(cx), float(cy)
        self.first_frame = self.last_frame = frame_idx
        self.history = [(frame_idx, float(cx), float(cy))]
        self.crop = crop  # Save the image crop of the object
    def update(self, cx, cy, frame_idx):
        self.cx, self.cy = float(cx), float(cy)
        self.last_frame = frame_idx
        self.history.append((frame_idx, float(cx), float(cy)))

def reset_track_ids():
    Track._next_id = 1

def update_tracks(tracks, detections, frame_idx, frame, max_dist=120.0, max_missed=30):
    """Match detections to tracks using nearest Euclidean distance."""
    det_centers = [(float(d[4]), float(d[5])) for d in detections]
    n_det, n_trk = len(det_centers), len(tracks)
    matched_det = [False] * n_det
    matched_trk = [False] * n_trk
    assignments = []

    if n_det and n_trk:
        pairs = []
        for di, (dx, dy) in enumerate(det_centers):
            for ti, trk in enumerate(tracks):
                d = np.hypot(dx - trk.cx, dy - trk.cy)
                if d <= max_dist:
                    pairs.append((d, di, ti))
        pairs.sort(key=lambda p: p[0])
        for d, di, ti in pairs:
            if not matched_det[di] and not matched_trk[ti]:
                matched_det[di] = matched_trk[ti] = True
                assignments.append((di, ti))

    for di, ti in assignments:
        tracks[ti].update(det_centers[di][0], det_centers[di][1], frame_idx)

    # Create new tracks for unmatched detections
    for di in range(n_det):
        if not matched_det[di]:
            x, y, w, h, cx, cy = detections[di]
            # Extract the object's image crop
            x1, y1 = max(0, int(x)), max(0, int(y))
            x2, y2 = min(frame.shape[1], int(x+w)), min(frame.shape[0], int(y+h))
            crop = frame[y1:y2, x1:x2].copy()
            
            tracks.append(Track(cx, cy, frame_idx, crop=crop))

    tracks = [t for t in tracks if (frame_idx - t.last_frame) <= max_missed]
    return tracks, assignments

# --- 1. Run the tracker ---
print("\nStarting Euclidean tracker...")
reset_track_ids()
tracks = []
all_tracks = {}

for i, frame in enumerate(frames1):
    dets, _ = detect_objects(frame, BG0_1, min_area=1500)
    tracks, _ = update_tracks(tracks, dets, i, frame, max_dist=120.0, max_missed=30)
    for t in tracks:
        all_tracks[t.id] = t

print(f"Tracking complete. Created {Track._next_id - 1} unique tracks.")

# --- 2. Generate the Synopsis Frames ---
# Filter out noise (keep only tracks that appeared for at least 10 frames)
valid_tracks = [t for t in all_tracks.values() if len(t.history) >= 10]
valid_tracks.sort(key=lambda t: t.first_frame)  # Sort by time of appearance

print(f"Generating synopsis for {len(valid_tracks)} valid tracks...")


# getting fps of the original video for timestamping
cap = cv2.VideoCapture('tabletennis_short.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print(f"Original video FPS: {fps}")


grid_cols = 4
grid_rows = 3

canvas_h, canvas_w = BG0_1.shape[:2]

cell_w = canvas_w // grid_cols
cell_h = canvas_h // grid_rows

canvas = cv2.cvtColor(BG0_1, cv2.COLOR_GRAY2BGR) # Use BG0 as the reference frame
count_in_canvas = 0
synopsis_frame_idx = 0

for t in valid_tracks:
    if t.crop is None or t.crop.size == 0:
        continue
        
    # If the canvas is full, save it and start a new one
    if count_in_canvas >= (grid_cols * grid_rows):
        synopsis_frame_idx += 1
        cv2.imwrite(f"Results/Synopsis/synopsis_{synopsis_frame_idx:04d}.png", canvas)
        canvas = cv2.cvtColor(BG0_1, cv2.COLOR_GRAY2BGR)
        count_in_canvas = 0

    # Calculate grid position
    row = count_in_canvas // grid_cols
    col = count_in_canvas % grid_cols
    
    # Resize crop while preserving its original aspect ratio
    max_w = int(cell_w * 0.8)
    max_h = int(cell_h * 0.8)

    crop_h, crop_w = t.crop.shape[:2]

    scale = min(max_w / crop_w, max_h / crop_h)

    new_w = max(1, int(crop_w * scale))
    new_h = max(1, int(crop_h * scale))

    crop_resized = cv2.resize(t.crop, (new_w, new_h))

    ch, cw = crop_resized.shape[:2]
    
    # FIX: Convert the grayscale crop to BGR so it matches the canvas channels
    crop_resized_bgr = cv2.cvtColor(crop_resized, cv2.COLOR_GRAY2BGR)
    ch, cw = crop_resized_bgr.shape[:2]
    
    # Center the crop in the grid cell
    start_x = col * cell_w + (cell_w - cw) // 2
    start_y = row * cell_h + (cell_h - ch) // 2 + 20 # Shift down to make room for text
    
    # Paste crop onto canvas
    canvas[start_y:start_y+ch, start_x:start_x+cw] = crop_resized_bgr
    
    # Add Timestamp above the object
    timestamp = t.first_frame / fps
    text = f"{timestamp:.2f}s"
    cv2.putText(canvas, text, (start_x, start_y - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    count_in_canvas += 1

# Save the final canvas if it has anything on it
if count_in_canvas > 0:
    synopsis_frame_idx += 1
    cv2.imwrite(f"Results/Synopsis/synopsis_{synopsis_frame_idx:04d}.png", canvas)

print(f"Saved {synopsis_frame_idx} synopsis frames to Results/Synopsis/")

# --- 3. Create the output video ---
print("Creating synopsis video...")
writeVideo('tabletennis_synopsis.mp4', 'Results/Synopsis/synopsis_%04d.png', fps=fps)
print("Synopsis video created successfully!")

## Task 2: Extend Pipeline with Video Object Segmentation with Memory (SAM2)

In [ ]:
# task 2: extend pipeline with SAM2